# 01 · Setup & Dataset Verification (Pre-Split)

**Your dataset is already split into train/val/test on Drive.**

This notebook:
1. Sets up the repo & mounts Drive
2. Verifies dataset structure on Drive
3. Checks image counts & detects corrupt files
4. Shows sample grid

> ✏️ **Only Cell 3 needs editing** — set your Drive dataset path.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────
!pip install -q scikit-learn matplotlib seaborn tqdm Pillow pandas
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118
print('✓ Dependencies installed')

In [ ]:
# ── Cell 2: Clone repo ───────────────────────────────────────────────
import os, sys

GITHUB_USER = 'musarashid49'
REPO_NAME   = 'Image-Classification-with-CNN'
REPO_DIR    = f'/content/{REPO_NAME}'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/{GITHUB_USER}/{REPO_NAME}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'✓ Working dir: {os.getcwd()}')

In [ ]:
# ── Cell 3: Mount Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive mounted')

In [ ]:
# ── Cell 4: ✏️ SET YOUR DRIVE DATASET PATH (only edit this) ──────────
# Your pre-split dataset on Drive should have this structure:
#   /path/on/drive/
#       train/class_name/img.jpg ...
#       val/class_name/img.jpg ...
#       test/class_name/img.jpg ...

DRIVE_DATASET_PATH = '/content/drive/MyDrive/dataset'   # ← EDIT THIS

# Verify it exists
from pathlib import Path
assert Path(DRIVE_DATASET_PATH).is_dir(), f'Dataset not found: {DRIVE_DATASET_PATH}'

# Check structure
splits = ['train', 'val', 'test']
for s in splits:
    sp = Path(DRIVE_DATASET_PATH) / s
    assert sp.is_dir(), f'Missing {s} split at {sp}'

print(f'✓ Dataset found at: {DRIVE_DATASET_PATH}')
print(f'✓ Has train/ val/ test/ subfolders')

In [ ]:
# ── Cell 5: Verify structure & image counts ──────────────────────────
from pathlib import Path
from config.config import CLASS_NAMES

VALID_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

print(f'\nClass names in config ({len(CLASS_NAMES)}):  {CLASS_NAMES}\n')

for split in ['train', 'val', 'test']:
    split_path = Path(DRIVE_DATASET_PATH) / split
    print(f'{split.upper()}:')
    total = 0
    for cls_dir in sorted(split_path.iterdir()):
        if not cls_dir.is_dir(): continue
        imgs = [f for f in cls_dir.rglob('*') if f.suffix.lower() in VALID_EXTS]
        n = len(imgs)
        status = '✓' if n > 0 else '✗ EMPTY'
        print(f"  {status}  {cls_dir.name:<35} {n:>4} images")
        total += n
    print(f"  Total: {total}\n")

In [ ]:
# ── Cell 6: Scan for corrupt images ──────────────────────────────────
from PIL import Image

corrupt = []
for split in ['train', 'val', 'test']:
    split_path = Path(DRIVE_DATASET_PATH) / split
    for cls_dir in split_path.iterdir():
        if not cls_dir.is_dir(): continue
        for img_path in cls_dir.rglob('*'):
            if img_path.suffix.lower() not in VALID_EXTS: continue
            try:
                with Image.open(img_path) as img: img.verify()
            except Exception as e:
                corrupt.append(str(img_path))

if corrupt:
    print(f'⚠  {len(corrupt)} corrupt image(s):')
    for p in corrupt[:20]: print(f'   {p}')
    if len(corrupt) > 20: print(f'   ... and {len(corrupt)-20} more')
else:
    print('✓ No corrupt images found')

In [ ]:
# ── Cell 7: Sample grid ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import random

random.seed(42)
os.makedirs('results/plots', exist_ok=True)

sample_classes = CLASS_NAMES[:4]  # first 4 classes
N = 5

fig, axes = plt.subplots(len(sample_classes), N, figsize=(3*N, 3*len(sample_classes)))
for ri, cls in enumerate(sample_classes):
    cls_path = Path(DRIVE_DATASET_PATH) / 'train' / cls
    imgs = [f for f in cls_path.rglob('*') if f.suffix.lower() in VALID_EXTS]
    sampled = random.sample(imgs, min(N, len(imgs)))
    for ci in range(N):
        ax = axes[ri][ci]; ax.axis('off')
        if ci < len(sampled):
            try:
                ax.imshow(np.array(Image.open(sampled[ci]).convert('RGB').resize((112,112))))
            except: ax.text(0.5,0.5,'ERR',ha='center')
        if ci == 0: ax.set_ylabel(cls, fontsize=8, rotation=0, labelpad=60, va='center')

plt.suptitle('Sample Images (4 classes × 5 images from train set)', fontsize=13)
plt.tight_layout(); plt.show()

print(f'\n✓ Verification complete.')
print(f'\nNext → open 02_colab_dataloader_test.ipynb')